In [1]:
import time
from functools import wraps


def timer(func):
    """
    Decorator that measures execution time of a function.

    Args:
        func (callable): Function to be decorated.

    Returns:
        callable: Wrapped function with timing logic.
    """

    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()

        result = func(*args, **kwargs)

        end = time.time()
        elapsed = end - start

        print(f"[TIMER] {func.__name__} executed in {elapsed:.6f} seconds")

        return result

    return wrapper


def logger(func):
    """
    Decorator that logs function name, arguments, and return value.

    Args:
        func (callable): Function to wrap.

    Returns:
        callable: Wrapped function with logging.
    """

    @wraps(func)
    def wrapper(*args, **kwargs):

        print(f"[LOGGER] Calling: {func.__name__}")
        print(f"[LOGGER] Args: {args}")
        print(f"[LOGGER] Kwargs: {kwargs}")

        result = func(*args, **kwargs)

        print(f"[LOGGER] Returned: {result}")

        return result

    return wrapper


def retry(max_attempts=3):
    """
    Decorator factory that retries a function if it raises an exception.

    Args:
        max_attempts (int): Maximum retry attempts.

    Returns:
        callable: Decorator
    """

    def decorator(func):

        @wraps(func)
        def wrapper(*args, **kwargs):

            attempts = 0

            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)

                except Exception as e:
                    attempts += 1

                    print(f"[RETRY] Attempt {attempts} failed: {e}")

                    if attempts == max_attempts:
                        print("[RETRY] Max attempts reached")
                        raise

        return wrapper

    return decorator

In [2]:
@timer
@logger
def add(a, b):
    return a + b


print(add(5, 3))

[LOGGER] Calling: add
[LOGGER] Args: (5, 3)
[LOGGER] Kwargs: {}
[LOGGER] Returned: 8
[TIMER] add executed in 0.000245 seconds
8


In [3]:
import random


@retry(max_attempts=3)
def unreliable_function():
    if random.random() < 0.7:
        raise ValueError("Random failure")
    return "Success"


print(unreliable_function())

[RETRY] Attempt 1 failed: Random failure
Success


In [4]:
@retry(3)
@timer
@logger
def divide(a, b):
    return a / b


divide(10, 2)

[LOGGER] Calling: divide
[LOGGER] Args: (10, 2)
[LOGGER] Kwargs: {}
[LOGGER] Returned: 5.0
[TIMER] divide executed in 0.000372 seconds


5.0